In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%writefile /content/drive/MyDrive/Brain_tumor/src/preprocessing.py

Overwriting /content/drive/MyDrive/Brain_tumor/src/preprocessing.py


In [3]:
import os
import cv2
import numpy as np
from sklearn.utils import shuffle
from collections import Counter
import kagglehub

In [4]:

path = kagglehub.dataset_download("preetviradiya/brian-tumor-dataset")
print("Path to dataset files:", path)

100%|██████████| 107M/107M [00:00<00:00, 210MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/preetviradiya/brian-tumor-dataset/versions/1


In [5]:

images_path = os.path.join(
    path,
    'Brain Tumor Data Set',
    'Brain Tumor Data Set',
    'Brain Tumor'
)

images_pathhealthy = os.path.join(
    path,
    'Brain Tumor Data Set',
    'Brain Tumor Data Set',
    'Healthy'
)

print("Tumor Path:", images_path)
print("Healthy Path:", images_pathhealthy)

Tumor Path: /root/.cache/kagglehub/datasets/preetviradiya/brian-tumor-dataset/versions/1/Brain Tumor Data Set/Brain Tumor Data Set/Brain Tumor
Healthy Path: /root/.cache/kagglehub/datasets/preetviradiya/brian-tumor-dataset/versions/1/Brain Tumor Data Set/Brain Tumor Data Set/Healthy


In [6]:

CONFIG = {
    "IMG_SIZE": 224,
    "TUMOR_LABEL": 1,
    "HEALTHY_LABEL": 0
}

tumor_path = images_path
healthy_path = images_pathhealthy

In [7]:

def process_images(folder_path, label):
    data = []
    labels = []
    count = 0

    for img_name in os.listdir(folder_path):
        try:
            img_path = os.path.join(folder_path, img_name)

            # Read image (grayscale)
            img = cv2.imread(img_path, 0)

            if img is None:
                continue

            # Resize
            img = cv2.resize(img, (CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]))

            # Normalize
            img = img.astype("float32") / 255.0

            data.append(img)
            labels.append(label)

            count += 1

        except Exception as e:
            print(f"Error processing {img_name}")

    return data, labels, count

In [8]:

print("Processing Tumor Images...")
tumor_data, tumor_labels, tumor_count = process_images(
    tumor_path, CONFIG["TUMOR_LABEL"]
)

print("Processing Healthy Images...")
healthy_data, healthy_labels, healthy_count = process_images(
    healthy_path, CONFIG["HEALTHY_LABEL"]
)

# Combine
data = tumor_data + healthy_data
labels = tumor_labels + healthy_labels

print(f"Tumor count: {tumor_count}")
print(f"Healthy count: {healthy_count}")

Processing Tumor Images...
Processing Healthy Images...
Tumor count: 2513
Healthy count: 2087


In [9]:

X = np.array(data, dtype="float32")
y = np.array(labels)

print("Shape before reshape:", X.shape)

# Add channel dimension
X = X.reshape(-1, CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"], 1)

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)

Shape before reshape: (4600, 224, 224)
Final X shape: (4600, 224, 224, 1)
Final y shape: (4600,)


In [10]:

X, y = shuffle(X, y, random_state=42)
print("Data shuffled")

Data shuffled


In [11]:

print("Min pixel value:", X.min())
print("Max pixel value:", X.max())

Min pixel value: 0.0
Max pixel value: 1.0


In [12]:

class_counts = Counter(y)
total = len(y)

print("\nClass Distribution:")
print(f"Tumor (1): {class_counts[1]}")
print(f"Healthy (0): {class_counts[0]}")

print("\nPercentage Distribution:")
print(f"Tumor: {class_counts[1]/total * 100:.2f}%")
print(f"Healthy: {class_counts[0]/total * 100:.2f}%")

imbalance_ratio = class_counts[1] / class_counts[0]

print("\nImbalance Ratio:", round(imbalance_ratio, 2))

if imbalance_ratio < 1.5:
    print("No significant imbalance detected → No resampling required")
else:
    print("Significant imbalance → Resampling may be needed")


Class Distribution:
Tumor (1): 2513
Healthy (0): 2087

Percentage Distribution:
Tumor: 54.63%
Healthy: 45.37%

Imbalance Ratio: 1.2
No significant imbalance detected → No resampling required


In [13]:

np.savez_compressed("brain_tumor_data.npz", X=X, y=y)

print("Data saved successfully!")

Data saved successfully!
